# A Shift-Reduce Parser for Arithmetic Expressions

In this notebook we implement a generic *shift reduce parser*.  The parse table that we use 
implements the following grammar for arithmetic expressions:
$$
  \begin{eqnarray*}
  \mathrm{expr}        & \rightarrow & \mathrm{expr}\;\;\texttt{'+'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{expr}\;\;\texttt{'-'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{product}                                    \\[0.2cm]
  \mathrm{product}     & \rightarrow & \mathrm{product}\;\;\texttt{'*'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{product}\;\;\texttt{'/'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{factor}                                     \\[0.2cm]
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}     \\
                       & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#load "ShiftReducePure.fsx"
open ShiftReducePure
open SLR
open GrammarSyntax

The scanner is imported from the pure recognizer and retains the original lexeme alongside each token kind.

In [ ]:
printfn "%A" (tokenize "1 + 2 * (3 - 4)")

In [ ]:
type ParseTree = Leaf of string | Node of string * ParseTree list

let parse tokens =
    parseBy ParseTable.table (fun token -> Leaf token.Lexeme)
        (fun rule children -> Node(rule.Head, children)) tokens

## Visualization

In [ ]:
#load "../Support/Visualization.fsx"

Convert the typed parse tree to the shared Graphviz representation.

In [ ]:
let rec toTree = function
    | Leaf value -> Visualization.Leaf value
    | Node(name, children) -> Visualization.Node(name, List.map toTree children)
let drawTree tree = Visualization.treeToDot (toTree tree)

## Testing

In [ ]:
let test text =
    match parse (tokenize text) with
    | Ok tree -> printfn "%s" (drawTree tree); Some tree
    | Error message -> printfn "%s" message; None

In [ ]:
let tree = test "1 + 2 * 3"
checkNotebook (tree.IsSome)

In [ ]:
checkNotebook ((test "1 + 2 * (3 - 4)").IsSome)

In [ ]:
checkNotebook ((test "1 + * 2").IsNone)